# 02 — Baseline YOLO

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics pycocotools faster-coco-eval

In [ ]:
import pandas as pd
import yaml
from ultralytics import YOLO, settings

from vod import CLASSES, SEED
from vod.data import find_data_root
from vod.evaluate import boxes_xyxy_to_xywh, coco_evaluate
from vod.submission import write_submission

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
settings.update({"wandb": False})

## Config

In [ ]:
CFG = dict(
    model="yolo11s.pt",
    weights=None,  # path to an existing best.pt to skip training

    imgsz=1280,
    epochs=20,
    batch=8,
    TRAIN_STEP=5,
    VAL_STEP=10,  # only for validation during training
    conf=0.001,
    iou=0.7,
    max_det=500,
)
CFG

## Data

In [ ]:
def frame_list(split, step):
    frames = []
    for clip in sorted((DATA / split / "images").iterdir()):
        frames += sorted(clip.glob("*.jpg"))[::step]
    return frames


train_frames = frame_list("train", CFG["TRAIN_STEP"])
val_frames = frame_list("val", CFG["VAL_STEP"])
(WORK / "train.txt").write_text("\n".join(map(str, train_frames)))
(WORK / "val.txt").write_text("\n".join(map(str, val_frames)))

data_yaml = WORK / "data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path": str(WORK), "train": "train.txt", "val": "val.txt", "names": dict(enumerate(CLASSES)),
}))
print(len(train_frames), "train frames,", len(val_frames), "val frames")

## Train

In [ ]:
if CFG["weights"]:
    best = Path(CFG["weights"])
else:
    model = YOLO(CFG["model"])
    model.train(
        data=str(data_yaml), imgsz=CFG["imgsz"], epochs=CFG["epochs"], batch=CFG["batch"],
        seed=SEED, deterministic=True, project=str(WORK / "runs"), name="baseline", exist_ok=True,
    )
    best = WORK / "runs" / "baseline" / "weights" / "best.pt"

## Validation (all val frames)

In [ ]:
def predict(model, frames):
    parts, ms = [], []
    for res in model.predict([str(f) for f in frames], imgsz=CFG["imgsz"], conf=CFG["conf"],
                             iou=CFG["iou"], max_det=CFG["max_det"], stream=True, verbose=False):
        ms.append(sum(res.speed.values()))
        boxes = res.boxes.cpu().numpy()
        part = pd.DataFrame(boxes_xyxy_to_xywh(boxes.xyxy), columns=["x", "y", "w", "h"], dtype="float32")
        part["cls"] = boxes.cls.astype("int8")
        part["score"] = boxes.conf.astype("float32")
        part["file_name"] = res.path
        parts.append(part)
    return pd.concat(parts, ignore_index=True), sum(ms) / len(ms)


model = YOLO(best)
all_val = frame_list("val", 1)
val_dets, ms_per_img = predict(model, all_val)
metrics = coco_evaluate(DATA / "val" / "annotations_coco.json", val_dets, file_names=all_val, per_class=True)
metrics["ms_per_image"] = ms_per_img
del val_dets
pd.Series(metrics).round(4).to_frame("baseline")

## Submission

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
path_to_id = {str(DATA / f): i for f, i in zip(test["file_name"], test["image_id"])}
test_dets, _ = predict(model, [DATA / f for f in test["file_name"]])
test_dets["image_id"] = test_dets.pop("file_name").map(path_to_id)

sub = write_submission(test_dets, DATA / "test_images.csv", WORK / "submission_baseline.csv")
sub.head()